In [1]:
# ============================================================

import pandas as pd
import numpy as np
import time
import os

# Machine Learning Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

# Evaluation Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Model persistence
import joblib

print("="*60)
print("NOTEBOOK 02: MODEL TRAINING & EVALUATION")
print("="*60)

NOTEBOOK 02: MODEL TRAINING & EVALUATION


In [2]:
# ============================================================
# STEP 1: LOAD PROCESSED DATA
# Load train/test splits from disk (created in Notebook 01)
# ============================================================

print("\n" + "="*50)
print("STEP 1: LOADING PROCESSED DATA")
print("="*50)

# Load features and targets
X_train = pd.read_csv('../data/processed/X_train.csv')
X_test = pd.read_csv('../data/processed/X_test.csv')
y_train = pd.read_csv('../data/processed/y_train.csv').squeeze()
y_test = pd.read_csv('../data/processed/y_test.csv').squeeze()

print(f"✅ Training set:   X_train {X_train.shape}, y_train {y_train.shape}")
print(f"✅ Test set:       X_test {X_test.shape}, y_test {y_test.shape}")
print(f"✅ Features: {list(X_train.columns[:5])}... ({len(X_train.columns)} total)")

# Verify data types (all should be numeric)
print(f"\nData types check:")
print(f"   Object columns: {X_train.select_dtypes(include='object').shape[1]}")
print(f"   Bool columns: {X_train.select_dtypes(include='bool').shape[1]}")
print(f"   All numeric: {(X_train.dtypes != 'object').all()}")


STEP 1: LOADING PROCESSED DATA
✅ Training set:   X_train (648360, 25), y_train (648360,)
✅ Test set:       X_test (196032, 25), y_test (196032,)
✅ Features: ['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'CompetitionDistance']... (25 total)

Data types check:
   Object columns: 0
   Bool columns: 0
   All numeric: True


In [3]:
# ============================================================
# STEP 2: CREATE SAMPLES FOR FAST EXPERIMENTATION
# Full dataset is large (~800K rows) - sample for quick iteration
# Final models can train on full data later if needed
# ============================================================

print("\n" + "="*50)
print("STEP 2: SAMPLING FOR SPEED")
print("="*50)

# Sample size for prototyping (adjust based on your computer speed)
SAMPLE_SIZE_TRAIN = 100000
SAMPLE_SIZE_TEST = 50000

# Create stratified random samples
X_train_sample = X_train.sample(n=SAMPLE_SIZE_TRAIN, random_state=42)
y_train_sample = y_train.loc[X_train_sample.index]

X_test_sample = X_test.sample(n=SAMPLE_SIZE_TEST, random_state=42)
y_test_sample = y_test.loc[X_test_sample.index]

print(f"✅ Training sample: {X_train_sample.shape[0]:,} rows")
print(f"✅ Test sample:     {X_test_sample.shape[0]:,} rows")
print(f"✅ Sampling ratio:  {SAMPLE_SIZE_TRAIN/len(X_train)*100:.1f}% of full data")


STEP 2: SAMPLING FOR SPEED
✅ Training sample: 100,000 rows
✅ Test sample:     50,000 rows
✅ Sampling ratio:  15.4% of full data


In [4]:
# ============================================================
# STEP 3: DEFINE EVALUATION FUNCTION
# Consistent metrics across all models with optional subsampling
# ============================================================

print("\n" + "="*50)
print("STEP 3: EVALUATION FUNCTION")
print("="*50)

def evaluate_model(y_true, y_pred, model_name, sample_size=50000):
    """
    Evaluates model performance using RMSE and MAE.
    
    Parameters:
    -----------
    y_true : array-like
        Actual sales values
    y_pred : array-like
        Predicted sales values
    model_name : str
        Name of the model being evaluated
    sample_size : int
        Maximum samples to use for fast evaluation (speed optimization)
    
    Returns:
    --------
    rmse, mae : float
        Root Mean Squared Error and Mean Absolute Error
    """
    # Convert to numpy for faster computation
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    
    # Subsample if data is huge (prevents slow evaluation)
    if len(y_true) > sample_size:
        np.random.seed(42)  # Reproducible
        idx = np.random.choice(len(y_true), sample_size, replace=False)
        y_true = y_true[idx]
        y_pred = y_pred[idx]
    
    # Calculate metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    # Print formatted results
    print(f"\n{'='*40}")
    print(f"{model_name} Performance")
    print(f"{'='*40}")
    print(f"RMSE: €{rmse:,.2f}")
    print(f"MAE:  €{mae:,.2f}")
    print(f"{'='*40}")
    
    return rmse, mae

print("✅ Evaluation function ready")


STEP 3: EVALUATION FUNCTION
✅ Evaluation function ready


In [5]:
# ============================================================
# STEP 4: BASELINE MODEL - LINEAR REGRESSION
# Simple baseline to compare advanced models against
# ============================================================

print("\n" + "="*50)
print("STEP 4: LINEAR REGRESSION (BASELINE)")
print("="*50)

# Initialize model
lr_model = LinearRegression()

# Train and time
start_time = time.time()
print("Training...")
lr_model.fit(X_train_sample, y_train_sample)
train_time = time.time() - start_time

print(f"✅ Training complete in {train_time:.2f} seconds")

# Generate predictions
print("Predicting...")
y_pred_lr = lr_model.predict(X_test_sample)

# Evaluate
rmse_lr, mae_lr = evaluate_model(y_test_sample, y_pred_lr, "Linear Regression")


STEP 4: LINEAR REGRESSION (BASELINE)
Training...
✅ Training complete in 0.31 seconds
Predicting...

Linear Regression Performance
RMSE: €2,678.76
MAE:  €2,001.17


In [6]:
# ============================================================
# STEP 5: RANDOM FOREST (ENSEMBLE MODEL)
# Strong baseline with better nonlinear handling
# ============================================================

print("\n" + "="*50)
print("STEP 5: RANDOM FOREST")
print("="*50)

# Initialize with reduced complexity for speed
rf_model = RandomForestRegressor(
    n_estimators=50,          # Number of trees (reduced for speed)
    max_depth=8,              # Limit depth to prevent overfitting
    min_samples_split=100,    # Minimum samples to split a node
    min_samples_leaf=50,      # Minimum samples in leaf node
    random_state=42,          # Reproducibility
    n_jobs=-1                 # Use all CPU cores
)

# Train and time
start_time = time.time()
print("Training...")
rf_model.fit(X_train_sample, y_train_sample)
train_time = time.time() - start_time

print(f"✅ Training complete in {train_time:.2f} seconds")

# Generate predictions
print("Predicting...")
y_pred_rf = rf_model.predict(X_test_sample)

# Evaluate
rmse_rf, mae_rf = evaluate_model(y_test_sample, y_pred_rf, "Random Forest")


STEP 5: RANDOM FOREST
Training...
✅ Training complete in 19.06 seconds
Predicting...

Random Forest Performance
RMSE: €2,445.98
MAE:  €1,776.82


In [7]:
# ============================================================
# STEP 6: XGBOOST (GRADIENT BOOSTING)
# Industry standard for tabular data - expected best performer
# ============================================================

print("\n" + "="*50)
print("STEP 6: XGBOOST")
print("="*50)

# Initialize XGBoost with balanced parameters
xgb_model = xgb.XGBRegressor(
    n_estimators=100,         # Boosting rounds
    learning_rate=0.1,        # Step size shrinkage
    max_depth=6,              # Tree depth
    subsample=0.8,            # Row sampling ratio (prevents overfitting)
    colsample_bytree=0.8,     # Feature sampling ratio
    random_state=42,          # Reproducibility
    n_jobs=-1,                # Use all CPU cores
    objective='reg:squarederror',  # Regression objective
    reg_alpha=0.1,            # L1 regularization
    reg_lambda=1.0            # L2 regularization
)

# Train and time
start_time = time.time()
print("Training...")
xgb_model.fit(
    X_train_sample, 
    y_train_sample,
    eval_set=[(X_test_sample, y_test_sample)],  # Validation set
    verbose=False
)
train_time = time.time() - start_time

print(f"✅ Training complete in {train_time:.2f} seconds")

# Generate predictions
print("Predicting...")
y_pred_xgb = xgb_model.predict(X_test_sample)

# Evaluate
rmse_xgb, mae_xgb = evaluate_model(y_test_sample, y_pred_xgb, "XGBoost")


STEP 6: XGBOOST
Training...
✅ Training complete in 9.05 seconds
Predicting...

XGBoost Performance
RMSE: €1,790.43
MAE:  €1,312.11


In [8]:
# ============================================================
# STEP 7: MODEL COMPARISON TABLE
# Rank all models by performance
# ============================================================

print("\n" + "="*50)
print("STEP 7: MODEL COMPARISON")
print("="*50)

# Create comparison dataframe
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'RMSE': [rmse_lr, rmse_rf, rmse_xgb],
    'MAE': [mae_lr, mae_rf, mae_xgb]
}).sort_values('RMSE')

print("\n" + "="*50)
print("FINAL RANKINGS (by RMSE - lower is better)")
print("="*50)
print(results.to_string(index=False))

# Calculate improvements
print(f"\n📊 Key Insights:")
print(f"   • XGBoost beats Linear Regression by {((rmse_lr - rmse_xgb)/rmse_lr*100):.1f}%")
print(f"   • XGBoost beats Random Forest by {((rmse_rf - rmse_xgb)/rmse_rf*100):.1f}%")
print(f"   • Winner: {results.iloc[0]['Model']} (RMSE: €{results.iloc[0]['RMSE']:,.2f})")


STEP 7: MODEL COMPARISON

FINAL RANKINGS (by RMSE - lower is better)
            Model        RMSE         MAE
          XGBoost 1790.427463 1312.107300
    Random Forest 2445.977347 1776.821983
Linear Regression 2678.764385 2001.172561

📊 Key Insights:
   • XGBoost beats Linear Regression by 33.2%
   • XGBoost beats Random Forest by 26.8%
   • Winner: XGBoost (RMSE: €1,790.43)


In [9]:
# ============================================================
# STEP 8: FEATURE IMPORTANCE ANALYSIS
# Understand what drives sales predictions
# ============================================================

print("\n" + "="*50)
print("STEP 8: FEATURE IMPORTANCE")
print("="*50)

# Random Forest feature importance
rf_importance = pd.DataFrame({
    'Feature': X_train_sample.columns,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n🌲 Random Forest - Top 10 Features:")
print(rf_importance.head(10).to_string(index=False))

# XGBoost feature importance
xgb_importance = pd.DataFrame({
    'Feature': X_train_sample.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\n⚡ XGBoost - Top 10 Features:")
print(xgb_importance.head(10).to_string(index=False))

# Store for visualization notebook
rf_importance.to_csv('../visualization/rf_importance.csv', index=False)
xgb_importance.to_csv('../visualization/xgb_importance.csv', index=False)
print("\n✅ Feature importance saved to visualization/")


STEP 8: FEATURE IMPORTANCE

🌲 Random Forest - Top 10 Features:
            Feature  Importance
              Promo    0.215481
            IsPromo    0.143984
CompetitionDistance    0.137592
          DayOfWeek    0.083805
    Promo2SinceYear    0.078217
         WeekOfYear    0.069037
        StoreType_b    0.048990
              Store    0.046963
       Assortment_c    0.042783
                Day    0.031501

⚡ XGBoost - Top 10 Features:
                 Feature  Importance
                 IsPromo    0.313175
                   Promo    0.128399
             StoreType_b    0.072154
                  Promo2    0.058358
            Assortment_c    0.043310
         Promo2SinceYear    0.037912
     CompetitionDistance    0.036739
CompetitionOpenSinceYear    0.030340
         Promo2SinceWeek    0.030085
                   Store    0.029872


OSError: Cannot save file into a non-existent directory: '..\visualization'

In [10]:
# ============================================================
# STEP 9: SAVE TRAINED MODELS
# Persist best models for deployment and future use
# ============================================================

print("\n" + "="*50)
print("STEP 9: SAVING MODELS")
print("="*50)

# Ensure models directory exists
os.makedirs('../models/', exist_ok=True)

# Save XGBoost (winner) with descriptive name
xgb_path = '../models/xgboost_sales_model.pkl'
joblib.dump(xgb_model, xgb_path)
xgb_size = os.path.getsize(xgb_path) / 1024 / 1024

print(f"✅ XGBoost saved: {xgb_path}")
print(f"   Size: {xgb_size:.2f} MB")

# Save Random Forest (backup/interpretability)
rf_path = '../models/random_forest_model.pkl'
joblib.dump(rf_model, rf_path)
rf_size = os.path.getsize(rf_path) / 1024 / 1024

print(f"✅ Random Forest saved: {rf_path}")
print(f"   Size: {rf_size:.2f} MB")

# Save Linear Regression (baseline reference)
lr_path = '../models/linear_regression_model.pkl'
joblib.dump(lr_model, lr_path)
lr_size = os.path.getsize(lr_path) / 1024 / 1024

print(f"✅ Linear Regression saved: {lr_path}")
print(f"   Size: {lr_size:.2f} MB")

print(f"\n📁 All models saved to: ../models/")


STEP 9: SAVING MODELS
✅ XGBoost saved: ../models/xgboost_sales_model.pkl
   Size: 0.46 MB
✅ Random Forest saved: ../models/random_forest_model.pkl
   Size: 0.90 MB
✅ Linear Regression saved: ../models/linear_regression_model.pkl
   Size: 0.00 MB

📁 All models saved to: ../models/


In [11]:
# ============================================================
# STEP 10: MODEL VALIDATION - TEST LOADING
# Verify saved models work correctly
# ============================================================

print("\n" + "="*50)
print("STEP 10: VALIDATION")
print("="*50)

# Load XGBoost and test
loaded_model = joblib.load('../models/xgboost_sales_model.pkl')
test_pred = loaded_model.predict(X_test_sample.iloc[:5])

print("✅ Model load test successful")
print(f"   Sample predictions: {test_pred}")
print(f"   Expected range: €4,000 - €8,000 (typical daily sales)")

# Verify predictions match original
original_pred = xgb_model.predict(X_test_sample.iloc[:5])
match = np.allclose(test_pred, original_pred)

print(f"\n✅ Save/Load integrity: {'PASS' if match else 'FAIL'}")


STEP 10: VALIDATION
✅ Model load test successful
   Sample predictions: [6358.275  4595.5522 6262.6367 5952.622  4406.5664]
   Expected range: €4,000 - €8,000 (typical daily sales)

✅ Save/Load integrity: PASS


In [13]:
# ============================================================
# STEP 11: OPTIONAL - SHAP EXPLAINABILITY
# Generate SHAP values if shap is installed
# ============================================================

print("\n" + "="*50)
print("STEP 11: SHAP EXPLAINABILITY (OPTIONAL)")
print("="*50)

try:
    import shap
    
    print("SHAP installed - generating explanations...")
    
    # Create explainer
    explainer = shap.TreeExplainer(xgb_model)
    
    # Calculate for sample (first 1000 for speed)
    shap_sample_size = 1000
    shap_values = explainer.shap_values(X_test_sample.iloc[:shap_sample_size])
    
    print(f"✅ SHAP values calculated for {shap_sample_size} samples")
    print(f"   Base value (expected): €{explainer.expected_value:,.2f}")
    
    # Save SHAP values for visualization notebook
    np.save('../visualization/shap_values.npy', shap_values)
    X_test_sample.iloc[:shap_sample_size].to_csv('../visualization/shap_sample.csv', index=False)
    
    print("✅ SHAP data saved for visualization")
    
except ImportError:
    print("⚠️ SHAP not installed")
    print("   Install with: pip install shap")
    print("   Then re-run this cell or use Notebook 03")


STEP 11: SHAP EXPLAINABILITY (OPTIONAL)
SHAP installed - generating explanations...
✅ SHAP values calculated for 1000 samples
   Base value (expected): €6,924.68
✅ SHAP data saved for visualization


In [15]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "="*60)
print("NOTEBOOK 02 COMPLETE")
print("="*60)

print(f"\n📊 Final Results:")
print(f"   Best Model: XGBoost")
print(f"   RMSE: €{rmse_xgb:,.2f}")
print(f"   MAE:  €{mae_xgb:,.2f}")

print(f"\n💾 Outputs Created:")
print(f"   • 3 saved models in ../models/")
print(f"   • Feature importance data in ../visualization/")
print(f"   • SHAP values (if generated)")

print("="*60)


NOTEBOOK 02 COMPLETE

📊 Final Results:
   Best Model: XGBoost
   RMSE: €1,790.43
   MAE:  €1,312.11

💾 Outputs Created:
   • 3 saved models in ../models/
   • Feature importance data in ../visualization/
   • SHAP values (if generated)
